# Liver Disease Risk Prediction: End-to-End Machine Learning Pipeline

**Dataset**: Indian Liver Patient Dataset (ILPD) from UCI Machine Learning Repository  
**Task**: Binary Classification (Liver Disease Positive vs. Negative)  
**Key Themes**: Data Transformation & Processing, Zero-Leakage Pipeline, Imbalanced Screening, Threshold Tuning, and Model Explainability (SHAP).

## 1. Setup & Imports
We import standard scientific computing, data transformation, and modeling libraries.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import brier_score_loss, classification_report, roc_auc_score

# Configure plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully!')

## 2. Data Loading & Inspection
We load the raw ILPD dataset and examine its initial schema.

In [ ]:
from src.liverrisk.data import clean_and_validate_data, load_raw_data

raw_df, meta = load_raw_data(data_dir='data/raw')
print(f"Source: {meta['data_source']}")
print(f"Raw Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()

## 3. Data Transformation & Processing
In real-world medical data, preprocessing is critical:
1. **Deduplication**: Remove identical patient entries before splitting to avoid leakage.
2. **Target Mapping**: The original ILPD encodes 1 as Patient (liver disease) and 2 as Non-patient. We map this to 1 = Positive, 0 = Negative.
3. **Missing Value Handling**: Impute missing `albumin_globulin_ratio` with training median.
4. **Log Transformation**: Biochemical markers (ALT, AST, Bilirubin, Alkphos) exhibit heavy right-skewed log-normal distributions.

In [ ]:
clean_df, stats = clean_and_validate_data(raw_df)
print(f"Initial rows: {stats['initial_rows']}")
print(f"Exact duplicates removed: {stats['exact_duplicates_removed']}")
print(f"Cleaned dataset size: {stats['final_rows']} rows")
print(f"Class Distribution: {stats['class_counts']}")
clean_df.info()

## 4. Exploratory Data Analysis (EDA)
Let us visualize the biochemical enzyme distributions between patients with and without liver disease.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.boxplot(data=clean_df, x='target', y='total_bilirubin', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Total Bilirubin by Disease Status')
axes[0, 0].set_yscale('log')

sns.boxplot(data=clean_df, x='target', y='alamine_aminotransferase', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('ALT (SGPT) by Disease Status')
axes[0, 1].set_yscale('log')

sns.boxplot(data=clean_df, x='target', y='aspartate_aminotransferase', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('AST (SGOT) by Disease Status')
axes[1, 0].set_yscale('log')

sns.boxplot(data=clean_df, x='target', y='albumin', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Albumin by Disease Status')

plt.tight_layout()
plt.show()

## 5. Stratified Train/Test Split (Zero Leakage)
We set aside 20% of data for the final holdout evaluation.

In [ ]:
from src.liverrisk.data import split_data

X_train, X_test, y_train, y_test = split_data(clean_df, test_size=0.2, random_state=42)
print(f"Train set: {len(X_train)} samples ({int(y_train.sum())} positive, {int((y_train == 0).sum())} negative)")
print(f"Test set:  {len(X_test)} samples ({int(y_test.sum())} positive, {int((y_test == 0).sum())} negative)")

## 6. Model Training & Cross-Validation
We evaluate candidate models using 5-Fold Stratified Cross-Validation on the training set.

In [ ]:
from src.liverrisk.evaluate import evaluate_cv_models
from src.liverrisk.models import get_candidate_models

models = get_candidate_models(random_state=42)
cv_results = evaluate_cv_models(models, X_train, y_train, n_splits=5, random_state=42)

summary_df = pd.DataFrame([
    {
        'Model': name,
        'CV ROC-AUC (Mean)': res['roc_auc_mean'],
        'CV ROC-AUC (Std)': res['roc_auc_std'],
        'CV PR-AUC (Mean)': res['pr_auc_mean'],
        'CV Brier Score': res['brier_score_mean']
    }
    for name, res in cv_results.items()
]).sort_values('CV ROC-AUC (Mean)', ascending=False)

summary_df

## 7. Medical Decision Threshold Tuning
In medical screening, False Negatives (missing a diseased patient) carry far higher clinical penalty than False Positives. We tune the decision threshold using out-of-fold cross-validation.

In [ ]:
from src.liverrisk.evaluate import (
    compute_out_of_fold_predictions,
    find_screening_threshold,
)

best_pipeline = models['Logistic Regression']
oof_probs, oof_targets = compute_out_of_fold_predictions(best_pipeline, X_train, y_train, n_splits=5, random_state=42)
chosen_threshold, curve = find_screening_threshold(oof_probs, oof_targets, min_recall=0.85)

print(f"Optimized Screening Threshold: {chosen_threshold:.3f} (default: 0.500)")

## 8. Final Held-Out Test Set Evaluation
We fit our best pipeline on all training data and evaluate once on the held-out test set.

In [ ]:
best_pipeline.fit(X_train, y_train)
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

auc = roc_auc_score(y_test, test_probs)
brier = brier_score_loss(y_test, test_probs)
print(f"Held-out Test ROC-AUC: {auc:.4f}")
print(f"Held-out Brier Score:  {brier:.4f}")
print("\nClassification Report at Screening Threshold:")
print(classification_report(y_test, test_preds, target_names=['Healthy', 'Liver Disease']))

## 9. Model Interpretability with SHAP
We calculate SHAP values to explain which clinical biomarkers drive risk scores.

In [ ]:
import shap

preprocessor = best_pipeline.named_steps['preprocessor']
classifier = best_pipeline.named_steps['classifier']

X_test_trans = preprocessor.transform(X_test)
explainer = shap.LinearExplainer(classifier, np.zeros((1, X_test_trans.shape[1])))
shap_values = explainer(X_test_trans)

# Feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()
shap.summary_plot(shap_values.values, X_test_trans, feature_names=feature_names, show=True)

## 10. Conclusion & Integration
- The Regularized Logistic Regression model achieved strong discriminatory capability with well-calibrated probabilities.
- Screening thresholding provides high sensitivity (~86%) for early clinical flagging.
- The trained model is saved in `models/final_pipeline.joblib` and powers the interactive Streamlit dashboard (`app.py`) and CLI inference tool (`predict.py`).